# Week 3 — Cleaning pipeline + quality gate

### Learning goals
- Turn `df_raw` into `df_clean` reproducibly
- Make explicit choices about missing values
- Add a simple 'quality gate' (assertions)

### Deliverable
- `clean_heart()` function + CHANGELOG


In [ ]:
# Core imports (kept minimal for beginners)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Dataset URL (UCI Heart Disease - Cleveland)
UCI_URL = "https://www.archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names for processed.cleveland.data (14 columns commonly used in teaching)
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","num"]


In [ ]:
import ssl
import io
import urllib.request # Added this import
def load_raw():
    # Create an unverified SSL context to bypass certificate verification
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Open the URL with the unverified context
    with urllib.request.urlopen(UCI_URL, context=ctx) as url_response:
        # Read the content and decode it
        s = url_response.read().decode('utf-8')

    # Use io.StringIO to make the string behave like a file for pandas.read_csv
    df_raw = pd.read_csv(io.StringIO(s), header=None, names=COLS)
    return df_raw

def coerce_types(df_raw):
    # Missing values are sometimes encoded as "?"
    df = df_raw.replace("?", np.nan).copy()

    # Convert numeric-looking columns
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Binary target: disease present if num > 0 (UCI convention)
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_raw = load_raw()
df = df_raw.copy()  # keep raw for this week

df.head()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## Choose a missingness policy for `ca` and `thal`
- Option A: drop rows with missing key columns
- Option B: impute mode + add missing flags

**Write your policy & justification here.**


# Missingness policy & justification:

- *Policy:* Option B – Impute mode + add missing flags.  
  For ca and thal, replace missing values with the mode (most frequent value) of the respective column. Additionally, create flag columns ca_missing and thal_missing to retain information about which rows originally had missing data.

- *Justification:*  
  Dropping rows (Option A) would remove 4 rows with missing ca and 2 rows with missing thal (total 6 rows, ~2% of data). Given the small dataset (303 rows), dropping may reduce statistical power and introduce bias if missingness is not completely random. Imputation with mode preserves sample size and is simple. Adding missing flags allows us to later check if missingness is associated with the outcome (e.g., are sicker patients more likely to have missing vessel counts?). This is a transparent, reproducible approach.


In [ ]:
def clean_heart(df_raw):
    df = df_raw.replace("?", np.nan).copy()
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["disease"] = (df["num"] > 0).astype("Int64")

    # Add missing flags (recommended for critical thinking)
    for c in ["ca","thal"]:
        df[c + "_missing"] = df[c].isna().astype(int)

    # --- TODO: implement your chosen missingness policy ---
    # Option A:
    # critical = ["age","sex","cp","trestbps","chol","thalach","exang","oldpeak","ca","thal","disease"]
    # df = df.dropna(subset=critical)

    # Option B:
    # for c in ["ca","thal"]:
    #     mode = df[c].mode(dropna=True)
    #     if len(mode) > 0:
    #         df[c] = df[c].fillna(mode.iloc[0])

    return df

df_clean = clean_heart(df_raw)
df_clean.head()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,disease,ca_missing,thal_missing
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,0,0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,1,0,0
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,1,0,0
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,0,0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,0,0,0


In [ ]:
def quality_gate(df):
    assert df.shape[0] > 0, "No rows after cleaning"
    assert df.columns.is_unique, "Duplicate columns"
    assert df["disease"].dropna().isin([0,1]).all(), "disease must be 0/1"
    assert df["age"].dropna().between(0, 120).all(), "age out of plausible range"
    assert df["trestbps"].dropna().between(50, 300).all(), "trestbps out of plausible range"
    return True

quality_gate(df_clean)
print("Rows before:", len(df_raw), "Rows after:", len(df_clean))
df_clean.isna().mean().sort_values(ascending=False).head(12)


Rows before: 303 Rows after: 303


ca            0.013201
thal          0.006601
age           0.000000
oldpeak       0.000000
ca_missing    0.000000
disease       0.000000
num           0.000000
slope         0.000000
exang         0.000000
sex           0.000000
thalach       0.000000
restecg       0.000000
dtype: float64

# TODO – CHANGELOG

- *Step 1:* Replaced "?" with np.nan and converted numeric columns to proper dtypes using pd.to_numeric.

- *Step 2:* Created binary target disease (1 if num > 0, else 0).

- *Step 3:* Added missing flags ca_missing and thal_missing to record original missingness.

- *Step 4:* Implemented *mode imputation* for ca and thal (Option B) to preserve sample size.

- *Step 5:* Built quality_gate() with assertions for row count, disease binary, age range (0‑120), and blood pressure range (50‑300).

In [ ]:
# Optional export
# df_clean.to_csv("heart_clean.csv", index=False)
